In [1]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, accuracy_score, mean_squared_error, r2_score
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import seaborn as sns

Train model neural network from cleaned_realistic_ocean_climate.csv

In [2]:
df_roc = pd.read_csv("../data/cleaned_realistic_ocean_climate.csv")

In [3]:
df_roc.shape

(500, 5)

In [4]:
df_roc.head()

,Latitude,Longitude,SST (°C),pH Level,Bleaching Severity
0,20.0248,38.4931,29.47,8.107,0
1,-18.2988,147.7782,29.65,8.004,2
2,14.9768,-75.0233,28.86,7.947,2
3,-18.3152,147.6486,28.97,7.995,1
4,-0.8805,-90.9769,28.60,7.977,0


In [5]:
X_roc = df_roc.drop(columns=["Bleaching Severity"])
y_roc = df_roc["Bleaching Severity"]
y_roc = y_roc.replace({2: 1})

In [6]:
X_roc.columns.tolist()
y_roc.value_counts()

Bleaching Severity
0    282
1    218
Name: count, dtype: int64

In [7]:
X_temp_roc, X_test_roc, y_temp_roc, y_test_roc = train_test_split( X_roc, y_roc, test_size=0.2, random_state=42, stratify=y_roc )

X_train_roc, X_val_roc, y_train_roc, y_val_roc = train_test_split( X_temp_roc, y_temp_roc, test_size=0.25, random_state=42, stratify=y_temp_roc )

In [8]:
scaler = MinMaxScaler()
X_train_scaled_roc = scaler.fit_transform(X_train_roc)
X_val_scaled_roc = scaler.transform(X_val_roc)
X_test_scaled_roc = scaler.transform(X_test_roc)

In [9]:
weights = class_weight.compute_class_weight( class_weight='balanced', classes=np.unique(y_train_roc), y=y_train_roc )
class_weights = dict(enumerate(weights))

In [10]:
num_features_roc = X_train_scaled_roc.shape[1]
num_classes_roc  = 1

In [11]:
model = keras.Sequential([
    layers.Input(shape=(num_features_roc,)),
    
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    layers.Dense(16, activation="relu"),
    
    layers.Dense(num_classes_roc, activation="sigmoid")
])

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329 (13.00 KB)

 Trainable params: 3,137 (12.25 KB)

 Non-trainable params: 192 (768.00 B)

In [13]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [14]:
callbacks = [
    keras.callbacks.EarlyStopping( monitor="val_loss", patience=10, restore_best_weights=True ),
    keras.callbacks.ReduceLROnPlateau( monitor="val_loss", factor=0.5, patience=5 )
]

In [15]:
history = model.fit(
    X_train_scaled_roc, y_train_roc,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_scaled_roc, y_val_roc),
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.5400 - loss: 0.9135 - val_accuracy: 0.4300 - val_loss: 0.6966 - learning_rate: 0.0010
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5067 - loss: 0.8414 - val_accuracy: 0.4200 - val_loss: 0.6991 - learning_rate: 0.0010
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5067 - loss: 0.8145 - val_accuracy: 0.4300 - val_loss: 0.6998 - learning_rate: 0.0010
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5400 - loss: 0.7820 - val_accuracy: 0.4300 - val_loss: 0.7010 - learning_rate: 0.0010
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5667 - loss: 0.7320 - val_accuracy: 0.4300 - val_loss: 0.7020 - learning_rate: 0.0010
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5567 - loss: 0.7227 - val_accuracy: 0.4300 - val_loss: 0.7030 - learning_rate: 0.0010
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4900 - loss: 0.7563 - val_ac

In [16]:
loss, accuracy = model.evaluate(X_test_scaled_roc, y_test_roc, verbose=0)
print(f"\n Test Accuracy: {accuracy:.4f}")
print(f" Test Loss: {loss:.4f}")

y_pred_prob_roc = model.predict(X_test_scaled_roc)
y_pred_roc = (y_pred_prob_roc > 0.4).astype(int)

print("\nClassification Report (Neural Network):")
print(classification_report(y_test_roc, y_pred_roc, target_names=["Low", "Severe"]))


 Test Accuracy: 0.4200
 Test Loss: 0.6968
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

Classification Report (Neural Network):
              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        56
      Severe       0.44      1.00      0.61        44

    accuracy                           0.44       100
   macro avg       0.22      0.50      0.31       100
weighted avg       0.19      0.44      0.27       100



C:\Users\Jirayut Pimmuen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Jirayut Pimmuen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Jirayut Pimmuen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-

In [17]:
rf_model_roc = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42
)

In [18]:
rf_model_roc.fit(X_train_scaled_roc, y_train_roc)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [19]:
y_pred_rf_roc = rf_model_roc.predict(X_test_scaled_roc)
print("\n Random Forest Test Accuracy:", accuracy_score(y_test_roc, y_pred_rf_roc))

print("\nClassification Report (Random Forest):")
print(classification_report(y_test_roc, y_pred_rf_roc, target_names=["Low", "Severe"]))


 Random Forest Test Accuracy: 0.53

Classification Report (Random Forest):
              precision    recall  f1-score   support

         Low       0.56      0.75      0.64        56
      Severe       0.44      0.25      0.32        44

    accuracy                           0.53       100
   macro avg       0.50      0.50      0.48       100
weighted avg       0.51      0.53      0.50       100



In [20]:
model.save("../models/neural_network_ROC.keras")
joblib.dump(rf_model_roc, "../models/random_forest_ROC.pkl")
joblib.dump(scaler, "../models/scaler_ROC.pkl")

['../models/scaler_ROC.pkl']

Train model neural network from cleaned_global_bleaching_events.csv

In [21]:
df_gbe = pd.read_csv("../data/cleaned_global_bleaching_events.csv")

In [22]:
df_gbe.shape

(34515, 8)

In [23]:
df_gbe.head()

,Depth_m,Distance_to_Shore,Turbidity,Windspeed,Cyclone_Frequency,Temperature_Maximum,SSTA_DHW,Percent_Bleaching
0,10.00,8519.23,0.0287,8.0,49.90,304.69,0.00,50.2
1,14.00,1431.62,0.0262,2.0,51.20,305.01,0.26,50.7
2,7.00,182.33,0.0429,8.0,61.52,304.14,0.00,50.9
3,9.02,313.13,0.0424,3.0,65.39,304.07,0.00,50.9
4,12.50,792.00,0.0424,3.0,65.39,303.76,0.00,50.9


In [24]:
x_gbe = df_gbe.drop(columns=["Percent_Bleaching"])
y_gbe = df_gbe["Percent_Bleaching"]

In [25]:
X_temp_gbe, X_test_gbe, y_temp_gbe, y_test_gbe = train_test_split( x_gbe, y_gbe, test_size=0.2, random_state=42 )

X_train_gbe, X_val_gbe, y_train_gbe, y_val_gbe = train_test_split( X_temp_gbe, y_temp_gbe, test_size=0.25, random_state=42 )

In [26]:
scaler_gbe = MinMaxScaler()
X_train_gbe_sc = scaler_gbe.fit_transform(X_train_gbe)
X_val_gbe_sc   = scaler_gbe.transform(X_val_gbe)
X_test_gbe_sc  = scaler_gbe.transform(X_test_gbe)

In [27]:
nn_gbe = keras.Sequential([
    layers.Input(shape=(X_train_gbe_sc.shape[1],)),

    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(16, activation="relu"),

    layers.Dense(1, activation="linear")
])
nn_gbe.compile(optimizer="adam", loss="huber", metrics=["mae"])

In [28]:
nn_gbe.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,801 (50.00 KB)

 Trainable params: 12,353 (48.25 KB)

 Non-trainable params: 448 (1.75 KB)

In [29]:
callbacks_gbe = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
]

In [30]:
y_train_log = np.log1p(y_train_gbe)
y_val_log   = np.log1p(y_val_gbe)

In [31]:
nn_gbe.fit(
    X_train_gbe_sc, y_train_log,
    epochs=100, batch_size=32,
    validation_data=(X_val_gbe_sc, y_val_log),
    callbacks=callbacks_gbe, verbose=0
)

In [32]:
print(y_gbe.min(), y_gbe.max())

0.0 100.0


In [33]:
y_pred_nn_gbe = np.expm1(nn_gbe.predict(X_test_gbe_sc).flatten())
rmse_nn = np.sqrt(mean_squared_error(y_test_gbe, y_pred_nn_gbe))
r2_nn   = r2_score(y_test_gbe, y_pred_nn_gbe)
print(f"✅ NN GBE → RMSE: {rmse_nn:.4f} | R²: {r2_nn:.4f}")

216/216 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
✅ NN GBE → RMSE: 19.7869 | R²: 0.0123


In [34]:
rf_reg_gbe = RandomForestRegressor(
    n_estimators=200, 
    max_depth=10,        # จำกัดความลึกนิดหน่อยป้องกัน Overfitting
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1            # ใช้ CPU เต็มกำลังให้รันเร็วขึ้น
)

In [35]:
rf_reg_gbe.fit(X_train_gbe_sc, y_train_log)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [36]:
y_pred_rf_log = rf_reg_gbe.predict(X_test_gbe_sc)
y_pred_rf_gbe = np.expm1(y_pred_rf_log)

In [37]:
rmse_rf = np.sqrt(mean_squared_error(y_test_gbe, y_pred_rf_gbe))
r2_rf = r2_score(y_test_gbe, y_pred_rf_gbe)
print(f"🌲 RF GBE → RMSE: {rmse_rf:.4f} | R²: {r2_rf:.4f}")

🌲 RF GBE → RMSE: 17.4716 | R²: 0.2299
